# SQD Workflow: Stage 1 (circuit construction) + Stage 2 (sampling) + Stage 2B (sbd export)

- **Stage 1** (`stage1.py`): Hamiltonian (FCIDUMP, from PySCF or supplied) -> CCSD -> LUCJ ansatz -> transpiled circuit -> `.qpy` + JSON metadata.
- **Stage 2** (`stage2.py`): loads the `.qpy`, samples with Qiskit Aer, writes bitstrings + counts and a JSON diagnostics summary.
- **Stage 2B** (`stage2b.py`): splits Stage 2's sampled bitstrings into the alpha/beta determinant files the downstream C++ app `sbd` (r-ccs-cms/sbd) actually consumes, plus an HF-only calibration file and a ready-to-paste run command.
- A single `CONFIG` dict (below) drives all three stages; `run_stage1(CONFIG)` / `run_stage2(CONFIG)` / `run_stage2b(CONFIG)` are importable for standalone scripts.
- Out of scope: sampling-quality metrics (Gini, compactness), orbital-ordering permutations, optimisation loops, and the diagonalisation itself (`sbd` does that).

In [1]:
from pathlib import Path

import stage1
import stage2
import stage2b

print("stage1, stage2, and stage2b modules loaded.")

stage1, stage2, and stage2b modules loaded.


## CONFIG

Single dict driving all three stages. `fcidump_source` selects Stage 1's
entry point: `"pyscf"` builds an FCIDUMP from the molecular parameters below;
`"existing"` loads `existing_fcidump_path` directly (set `atom`/`basis` to
`None` in that case - they are only used for metadata bookkeeping).

`preset` selects a named system from `stage1.PRESETS` (`n2_equilibrium`,
`n2_stretched`, `n2_very_stretched`, `h2_sto3g`) which fills in
`atom`/`basis`/active-space fields - but only where the value here is still
`None`; set any of those fields explicitly to override the preset. This cell
runs `n2_equilibrium` as the primary demonstration; the cross-preset
comparison and the H2 calibration test further down use their own copies.

The `sbd_*` / `*_determinants` / `write_bdetfile` keys configure Stage 2B's
export to the downstream C++ app's `--adetfile`/`--bdetfile` format - see
`stage2b.py`'s module docstring for what that app expects.

In [2]:
CONFIG = {
    # --- Stage 1: Hamiltonian source ---
    "fcidump_source": "pyscf",  # "pyscf" or "existing"
    "existing_fcidump_path": None,  # used only when fcidump_source == "existing"

    # --- Stage 1: system preset (see stage1.PRESETS) ---
    "preset": "n2_equilibrium",

    # --- Stage 1: molecular parameters (Path A). None => filled in from `preset`. ---
    "atom": None,
    "basis": None,
    "charge": 0,
    "spin": 0,
    "n_frozen_core": None,
    "n_active_orbitals": None,
    "n_active_electrons": None,

    # --- Stage 1: CCSD / CASCI failsafe + exact-wavefunction baseline ---
    "run_casci_reference": True,
    "casci_reference_max_norb": 12,

    # --- Stage 1: circuit construction ---
    "optimization_level": 3,
    # Builds a second, unmasked (interaction_pairs=None) LUCJ ansatz alongside
    # the masked one, to quantify what the locality mask costs. See
    # stage1.build_unmasked_lucj_operator / print_ansatz_comparison.
    "build_unmasked_comparison": True,

    # --- Stage 1: output locations ---
    "output_dir": "outputs",
    "fcidump_filename": "n2_cas66.fcidump",
    "circuit_filename": "n2_cas66_lucj.qpy",

    # --- Stage 2: sampling ---
    # SQD literature commonly uses 1e5-1e7 shots; 10,000 is too few to expose
    # rare correlated configurations, so the default here is 1,000,000.
    "shots": 1_000_000,
    "seed": 42,

    # --- Stage 2: output locations ---
    "bitstring_filename": "n2_cas66_bitstrings.txt",

    # --- Stage 2: bitstring file format (this is a separate artefact from the
    # Stage 2B sbd export below; it may keep its header/counts for analysis) ---
    "bit_order_mode": "qiskit",  # Stage 2B requires this to stay 'qiskit'
    "output_separator": " ",
    "include_header": True,
    "include_counts": True,
    # When True, write one bitstring file per bit-order mode (mode in the
    # filename) so all four can be tried against other consumers at once.
    "emit_all_bit_order_variants": False,

    # --- Stage 2: diagnostics ---
    "top_k_report": 10,

    # --- Stage 2B: sbd export (see stage2b.py) ---
    # 'split' is CONFIRMED correct (verified against the sbd repo's
    # AlphaDets.txt example file - see stage2b module docstring). The
    # alternatives remain available as a fallback only.
    "sbd_bit_transform": "split",  # 'split' | 'split_reverse' | 'split_swap'
    # When True, write adet/bdet files for all three transforms at once.
    "emit_all_sbd_variants": False,
    # sbd defaults beta=alpha when --bdetfile is omitted; set False to skip
    # writing the beta file and rely on that default.
    "write_bdetfile": True,
    # Keep only the N most frequently sampled alpha/beta determinants (each
    # side independently). None = keep all. Matters because sbd's Hilbert
    # space is the *product* of the alpha and beta sets, so it grows as N^2.
    "max_determinants": None,
}

print("CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

CONFIG:
  fcidump_source: pyscf
  existing_fcidump_path: None
  preset: n2_equilibrium
  atom: None
  basis: None
  charge: 0
  spin: 0
  n_frozen_core: None
  n_active_orbitals: None
  n_active_electrons: None
  run_casci_reference: True
  casci_reference_max_norb: 12
  optimization_level: 3
  build_unmasked_comparison: True
  output_dir: outputs
  fcidump_filename: n2_cas66.fcidump
  circuit_filename: n2_cas66_lucj.qpy
  shots: 1000000
  seed: 42
  bitstring_filename: n2_cas66_bitstrings.txt
  bit_order_mode: qiskit
  output_separator:  
  include_header: True
  include_counts: True
  emit_all_bit_order_variants: False
  top_k_report: 10
  sbd_bit_transform: split
  emit_all_sbd_variants: False
  write_bdetfile: True
  max_determinants: None


## Stage 1: circuit construction

Builds/loads the FCIDUMP, analyses the exact (CASCI) wavefunction's
concentration as a physical baseline, runs CCSD in the active space (with a
loud failsafe warning if CCSD diverges relative to CASCI), builds the masked
LUCJ operator (`n_reps=None`, `optimize=False` - mandatory) plus an unmasked
comparison ansatz, transpiles both, prints a masked-vs-unmasked comparison
table, and serialises both to `.qpy` + JSON metadata (the masked circuit's
metadata records the unmasked `.qpy` path so Stage 2 can find it). Every step
prints its own diagnostics.

In [3]:
stage1_result = stage1.run_stage1(CONFIG)

print()
print("Stage 1 outputs:")
print(f"  FCIDUMP         : {stage1_result.fcidump_path}")
print(f"  QPY (masked)    : {stage1_result.qpy_path}")
print(f"  Metadata (masked): {stage1_result.metadata_path}")
if stage1_result.unmasked_stats is not None:
    print(f"  QPY (unmasked)  : {stage1_result.unmasked_stats.qpy_path}")
    print(f"  Metadata (unmasked): {stage1_result.unmasked_stats.metadata_path}")

Active preset: 'n2_equilibrium'
=== Stage 1 / Path A: generating FCIDUMP from PySCF ===
  HF energy               = -108.8677463470
  Active space orbitals   = 6
  Active space electrons  = (3, 3) (alpha, beta)
  Frozen core orbitals    = 4
  FCIDUMP written to      = outputs/n2_cas66.fcidump
Reloading active-space Hamiltonian from outputs/n2_cas66.fcidump ...
Parsing outputs/n2_cas66.fcidump
Parsing outputs/n2_cas66.fcidump
  norb (active) = 6
  nelec (active) = (3, 3)
  core energy = -97.5489212436
  active-space RHF energy = -108.8677463470
Analysing exact (CASCI) wavefunction concentration ...
  CI space dimension            = 400
  Weight on top determinant     = 0.939267
  Determinants for 90% weight   = 1
  Determinants for 99% weight   = 16
  Determinants for 99.9% weight = 24
Running CCSD in the active space ...
  CCSD converged      = True
  CCSD total energy   = -108.9459113987
  CCSD correlation E  = -0.0781650517
Computing CASCI (exact diagonalisation) reference in the act

  CASCI (exact) energy = -108.9467159424


Locality mask (heavy-hex zig-zag layout):
  aa (same-spin) pairs retained: 5 / 15 possible -> [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)]
  ab (opposite-spin) pairs retained: 2 / 36 possible -> [(0, 0), (4, 4)]
Building LUCJ operator (n_reps=None, optimize=False; MANDATORY, do not change) ...
  Actual n_reps chosen by full-rank factorisation: 18
Building Qiskit circuit (HF prep + UCJ + measure_all) ...
  Circuit built: 12 qubits, depth 3
Pre-transpilation:  depth=3, two-qubit gates=0


Transpilation path: preset pass manager with ffsim.qiskit.PRE_INIT pre-init stage.
Post-transpilation: depth=372, two-qubit gates=1196
Masked ansatz variational energy: -108.8983064675
Building UNMASKED LUCJ operator (interaction_pairs=None; comparison only) ...
  Actual n_reps chosen by full-rank factorisation: 18
Building Qiskit circuit (HF prep + UCJ + measure_all) ...
  Circuit built: 12 qubits, depth 3
Pre-transpilation:  depth=3, two-qubit gates=0


Transpilation path: preset pass manager with ffsim.qiskit.PRE_INIT pre-init stage.
Post-transpilation: depth=517, two-qubit gates=1720
Unmasked ansatz variational energy: -108.9454183879

=== Masked vs unmasked LUCJ ansatz comparison ===
                                        masked          unmasked
n_reps                                      18                18
circuit depth                              372               517
two-qubit gate count                      1196              1720
variational energy             -108.8983064675   -108.9454183879
CCSD energy (reference)        -108.9459113987   -108.9459113987
CASCI energy (reference)       -108.9467159424   -108.9467159424
error vs CCSD (mHa)                    47.6049            0.4930

Circuit serialised to outputs/n2_cas66_lucj_unmasked.qpy
Metadata written to outputs/n2_cas66_lucj_unmasked.json
Validating round-trip of outputs/n2_cas66_lucj_unmasked.qpy ...
  Round-trip validation PASSED: qubits, depth, and operation cou

## Stage 2: sampling

Loads the masked `.qpy` circuit produced above, samples it with
`AerSimulator` (prints wall-clock sampling time), prints all diagnostics
(unique bitstring count, top-10, particle-number checks). If Stage 1 built an
unmasked comparison circuit, its path is read from the masked circuit's
metadata and it is sampled too, with a side-by-side sampling-diversity
comparison (unique bitstring count, top-1 fraction) printed at the end.
Writes the bitstring file(s) + JSON summary for each circuit sampled.

`CONFIG["qpy_path"]` is set from Stage 1's result so this cell can also be
re-run standalone against any previously produced `.qpy` file.

In [4]:
CONFIG["qpy_path"] = str(stage1_result.qpy_path)

stage2_result = stage2.run_stage2(CONFIG)

Loading metadata from outputs/n2_cas66_lucj.json ...
  norb = 6, nelec = [3, 3]
--- Sampling MASKED circuit ---
Loading circuit from outputs/n2_cas66_lucj.qpy ...
  qubits = 12, depth = 372
Sampling with AerSimulator: shots=1000000, seed=42 ...


  Got 50 unique bitstrings from 1000000 shots in 0.32s.


Total shots: 1000000
Unique bitstrings: 50
Top 10 bitstrings:
  000111000111  count=993991  prob=0.993991
  011001000111  count=817  prob=0.000817
  000111011001  count=805  prob=0.000805
  000111010011  count=558  prob=0.000558
  010011000111  count=545  prob=0.000545
  001101001101  count=426  prob=0.000426
  010101010101  count=393  prob=0.000393
  010011010011  count=314  prob=0.000314
  001011001011  count=195  prob=0.000195
  010011001011  count=189  prob=0.000189
Fraction of shots on the single most frequent bitstring: 0.993991
Particle-number check (fraction of shots, by weight):
  total set bits == 6: 100.0000%
  alpha set bits == 3: 100.0000%
  beta set bits == 3: 100.0000%


Bitstring file written to outputs/n2_cas66_bitstrings.txt (50 unique bitstrings)
JSON summary written to outputs/n2_cas66_bitstrings.json
--- Sampling UNMASKED circuit ---
Loading circuit from outputs/n2_cas66_lucj_unmasked.qpy ...
  qubits = 12, depth = 517
Sampling with AerSimulator: shots=1000000, seed=42 ...


  Got 78 unique bitstrings from 1000000 shots in 0.34s.


Total shots: 1000000
Unique bitstrings: 78
Top 10 bitstrings:
  000111000111  count=940770  prob=0.940770
  001011001011  count=8965  prob=0.008965
  010101010101  count=8929  prob=0.008929
  000111011001  count=3374  prob=0.003374
  011001000111  count=3319  prob=0.003319
  010101001011  count=2802  prob=0.002802
  001011010101  count=2749  prob=0.002749
  010011010101  count=2627  prob=0.002627
  010011001011  count=2539  prob=0.002539
  001011010011  count=2490  prob=0.002490
Fraction of shots on the single most frequent bitstring: 0.940770
Particle-number check (fraction of shots, by weight):
  total set bits == 6: 100.0000%
  alpha set bits == 3: 100.0000%
  beta set bits == 3: 100.0000%
Bitstring file written to outputs/n2_cas66_bitstrings_unmasked.txt (78 unique bitstrings)
JSON summary written to outputs/n2_cas66_bitstrings_unmasked.json

=== Masked vs unmasked sampling diversity ===
                                masked      unmasked
unique bitstrings                   50   

## Stage 2B: sbd export

Splits Stage 2's sampled bitstrings into the alpha/beta determinant files
the downstream C++ app (`sbd`, `apps/chemistry_tpb_selected_basis_diagonalization`)
actually consumes via `--adetfile`/`--bdetfile`: one bitstring per line, no
header, no counts. Prints the product-space diagnostics (sbd forms the
tensor product of the alpha and beta sectors - confirmed in the sbd README),
applies `CONFIG["max_determinants"]` truncation if set, and guarantees the
Hartree-Fock determinant is present in both files (sbd uses it as the
Davidson initial state after sorting).

Also writes an HF-only calibration file and prints a ready-to-paste `mpirun`
command for the files this run produced.

In [5]:
sbd_export = stage2b.run_stage2b(CONFIG)
hf_calibration = stage2b.write_hf_only_determinant(CONFIG)
stage2b.print_sbd_command(CONFIG, sbd_export)

=== Stage 2B: sbd export ===
Loaded 50 unique full bitstrings from outputs/n2_cas66_bitstrings.txt
Loading metadata from outputs/n2_cas66_lucj.json ...
  norb = 6, nelec = [3, 3]
--- sbd export: sbd_bit_transform='split' ---
  alpha determinants kept: 16/16 (100.0000% of sampled weight)
  beta determinants kept:  15/15 (100.0000% of sampled weight)
Determinant file written to outputs/n2_cas66_adets.txt (16 determinants)
Determinant file written to outputs/n2_cas66_bdets.txt (15 determinants)
Product-space diagnostics (tensor product of alpha x beta sectors):
  unique full bitstrings sampled  : 50
  unique alpha determinants       : 16
  unique beta determinants        : 15
  implied product-space dimension : 240 (= 16 x 15)
  product_dim / unique_full_bitstrings ratio: 4.80x
  full CI space dimension (C(norb,na)*C(norb,nb)): 400
  product space as % of full CI space: 60.0000%
sbd export diagnostics written to outputs/n2_cas66_sbd_export.json
=== Stage 2B complete ===
Loading metadata f

## H2 calibration test: the smallest possible end-to-end run

`h2_sto3g` (H2 at 0.74 A, sto-3g, no frozen core, norb=2, nelec=(1,1)) is the
smallest system this pipeline can build - useful for a fast, cheap sanity
check of the whole chain, and specifically for calibrating the sbd bit
ordering: `write_hf_only_determinant` below writes a single-determinant file
that, fed to sbd, must reproduce the printed Hartree-Fock energy exactly. If
it doesn't, try `CONFIG["sbd_bit_transform"] = "split_reverse"` or
`"split_swap"` (though `"split"` is already confirmed correct against the
sbd repository's own example file - see `stage2b.py`).

This runs Stage 1 -> Stage 2 -> Stage 2B end to end on its own `CONFIG` copy
so it doesn't disturb the primary `n2_equilibrium` outputs above.

In [6]:
h2_config = dict(CONFIG)  # fresh copy, same CONFIG pattern
h2_config["preset"] = "h2_sto3g"
h2_config["atom"] = None
h2_config["basis"] = None
h2_config["n_frozen_core"] = None
h2_config["n_active_orbitals"] = None
h2_config["n_active_electrons"] = None
h2_config["fcidump_filename"] = "h2_sto3g.fcidump"
h2_config["circuit_filename"] = "h2_sto3g_lucj.qpy"
h2_config["bitstring_filename"] = "h2_sto3g_bitstrings.txt"

h2_stage1 = stage1.run_stage1(h2_config)
h2_config["qpy_path"] = str(h2_stage1.qpy_path)
h2_stage2 = stage2.run_stage2(h2_config)
h2_sbd_export = stage2b.run_stage2b(h2_config)
h2_hf_calibration = stage2b.write_hf_only_determinant(h2_config)
stage2b.print_sbd_command(h2_config, h2_sbd_export)

Active preset: 'h2_sto3g'
=== Stage 1 / Path A: generating FCIDUMP from PySCF ===
  HF energy               = -1.1167593074
  Active space orbitals   = 2
  Active space electrons  = (1, 1) (alpha, beta)
  Frozen core orbitals    = 0
  FCIDUMP written to      = outputs/h2_sto3g.fcidump
Reloading active-space Hamiltonian from outputs/h2_sto3g.fcidump ...
Parsing outputs/h2_sto3g.fcidump
Parsing outputs/h2_sto3g.fcidump
  norb (active) = 2
  nelec (active) = (1, 1)
  core energy = 0.7151043391
  active-space RHF energy = -1.1167593074
Analysing exact (CASCI) wavefunction concentration ...
  CI space dimension            = 4
  Weight on top determinant     = 0.987334
  Determinants for 90% weight   = 1
  Determinants for 99% weight   = 2
  Determinants for 99.9% weight = 2
Running CCSD in the active space ...
  CCSD converged      = True
  CCSD total energy   = -1.1372839986
  CCSD correlation E  = -0.0205246912
Computing CASCI (exact diagonalisation) reference in the active space ...
  CA

  Got 4 unique bitstrings from 1000000 shots in 0.19s.
Total shots: 1000000
Unique bitstrings: 4
Top 4 bitstrings:
  0101  count=998459  prob=0.998459
  1010  count=766  prob=0.000766
  0110  count=390  prob=0.000390
  1001  count=385  prob=0.000385
Fraction of shots on the single most frequent bitstring: 0.998459
Particle-number check (fraction of shots, by weight):
  total set bits == 2: 100.0000%
  alpha set bits == 1: 100.0000%
  beta set bits == 1: 100.0000%
Bitstring file written to outputs/h2_sto3g_bitstrings.txt (4 unique bitstrings)
JSON summary written to outputs/h2_sto3g_bitstrings.json
--- Sampling UNMASKED circuit ---
Loading circuit from outputs/h2_sto3g_lucj_unmasked.qpy ...
  qubits = 4, depth = 17
Sampling with AerSimulator: shots=1000000, seed=42 ...
  Got 2 unique bitstrings from 1000000 shots in 0.19s.
Total shots: 1000000
Unique bitstrings: 2
Top 2 bitstrings:
  0101  count=987482  prob=0.987482
  1010  count=12518  prob=0.012518
Fraction of shots on the single mos

## Cross-preset comparison: quantifying HF-dominance vs. the locality mask

The single-preset run above showed the sampler heavily concentrated on one
bitstring. Two candidate explanations were proposed:

  (a) N2 in this active space is genuinely HF-dominated at this geometry
  (b) the locality mask discards most Jastrow elements, collapsing the
      ansatz back toward HF

This section runs Stage 1 + Stage 2 for all three presets
(`n2_equilibrium`, `n2_stretched`, `n2_very_stretched`), stretching the N-N
bond to reduce HF character (a), while the masked-vs-unmasked ansatz
comparison built into each Stage 1 run isolates the mask's own cost (b).
Each preset gets its own output filenames so nothing is overwritten.

Note: `n2_very_stretched` is expected to trip the CCSD-vs-CASCI divergence
failsafe (single-reference CCSD breaks down for stretched N2) - the loud
warning is expected there, and its ansatz/sampling numbers should be read as
"unreliable input" rather than a mask-quality measurement.

In [7]:
preset_results: dict[str, dict] = {}

for preset_name in ["n2_equilibrium", "n2_stretched", "n2_very_stretched"]:
    print()
    print("#" * 78)
    print(f"# Preset: {preset_name}")
    print("#" * 78)

    preset_config = dict(CONFIG)  # fresh copy per preset; same CONFIG pattern
    preset_config["preset"] = preset_name
    preset_config["atom"] = None
    preset_config["basis"] = None
    preset_config["n_frozen_core"] = None
    preset_config["n_active_orbitals"] = None
    preset_config["n_active_electrons"] = None
    preset_config["fcidump_filename"] = f"{preset_name}.fcidump"
    preset_config["circuit_filename"] = f"{preset_name}_lucj.qpy"
    preset_config["bitstring_filename"] = f"{preset_name}_bitstrings.txt"

    s1 = stage1.run_stage1(preset_config)
    preset_config["qpy_path"] = str(s1.qpy_path)
    s2 = stage2.run_stage2(preset_config)
    s2b = stage2b.run_stage2b(preset_config)

    preset_results[preset_name] = {"stage1": s1, "stage2": s2, "stage2b": s2b}

print()
print("Cross-preset comparison complete.")


##############################################################################
# Preset: n2_equilibrium
##############################################################################
Active preset: 'n2_equilibrium'
=== Stage 1 / Path A: generating FCIDUMP from PySCF ===


  HF energy               = -108.8677463470
  Active space orbitals   = 6
  Active space electrons  = (3, 3) (alpha, beta)
  Frozen core orbitals    = 4
  FCIDUMP written to      = outputs/n2_equilibrium.fcidump
Reloading active-space Hamiltonian from outputs/n2_equilibrium.fcidump ...
Parsing outputs/n2_equilibrium.fcidump
Parsing outputs/n2_equilibrium.fcidump
  norb (active) = 6
  nelec (active) = (3, 3)
  core energy = -97.5489212436
  active-space RHF energy = -108.8677463470
Analysing exact (CASCI) wavefunction concentration ...
  CI space dimension            = 400
  Weight on top determinant     = 0.939267
  Determinants for 90% weight   = 1
  Determinants for 99% weight   = 7
  Determinants for 99.9% weight = 25
Running CCSD in the active space ...
  CCSD converged      = True
  CCSD total energy   = -108.9459113976
  CCSD correlation E  = -0.0781650506
Computing CASCI (exact diagonalisation) reference in the active space ...
  CASCI (exact) energy = -108.9467159424
Locality m

Transpilation path: preset pass manager with ffsim.qiskit.PRE_INIT pre-init stage.
Post-transpilation: depth=371, two-qubit gates=1196
Masked ansatz variational energy: -108.8994796168
Building UNMASKED LUCJ operator (interaction_pairs=None; comparison only) ...
  Actual n_reps chosen by full-rank factorisation: 18
Building Qiskit circuit (HF prep + UCJ + measure_all) ...
  Circuit built: 12 qubits, depth 3
Pre-transpilation:  depth=3, two-qubit gates=0
Transpilation path: preset pass manager with ffsim.qiskit.PRE_INIT pre-init stage.
Post-transpilation: depth=516, two-qubit gates=1720
Unmasked ansatz variational energy: -108.9454183881

=== Masked vs unmasked LUCJ ansatz comparison ===
                                        masked          unmasked
n_reps                                      18                18
circuit depth                              371               516
two-qubit gate count                      1196              1720
variational energy             -108.89947961

  Got 48 unique bitstrings from 1000000 shots in 0.32s.
Total shots: 1000000
Unique bitstrings: 48
Top 10 bitstrings:
  000111000111  count=994115  prob=0.994115
  011001000111  count=816  prob=0.000816
  000111011001  count=805  prob=0.000805
  010011010011  count=643  prob=0.000643
  001101001101  count=576  prob=0.000576
  010101000111  count=276  prob=0.000276
  000111010101  count=271  prob=0.000271
  001011000111  count=255  prob=0.000255
  000111001011  count=252  prob=0.000252
  001011010101  count=146  prob=0.000146
Fraction of shots on the single most frequent bitstring: 0.994115
Particle-number check (fraction of shots, by weight):
  total set bits == 6: 100.0000%
  alpha set bits == 3: 100.0000%
  beta set bits == 3: 100.0000%
Bitstring file written to outputs/n2_equilibrium_bitstrings.txt (48 unique bitstrings)
JSON summary written to outputs/n2_equilibrium_bitstrings.json
--- Sampling UNMASKED circuit ---
Loading circuit from outputs/n2_equilibrium_lucj_unmasked.qpy ...
 

  Got 61 unique bitstrings from 1000000 shots in 0.34s.
Total shots: 1000000
Unique bitstrings: 61
Top 10 bitstrings:
  000111000111  count=940770  prob=0.940770
  010011010011  count=15426  prob=0.015426
  001101001101  count=15367  prob=0.015367
  010011001101  count=6999  prob=0.006999
  001101010011  count=6770  prob=0.006770
  000111011001  count=3374  prob=0.003374
  011001000111  count=3319  prob=0.003319
  001110001110  count=1260  prob=0.001260
  010110010110  count=1228  prob=0.001228
  011001011001  count=762  prob=0.000762
Fraction of shots on the single most frequent bitstring: 0.940770
Particle-number check (fraction of shots, by weight):
  total set bits == 6: 100.0000%
  alpha set bits == 3: 100.0000%
  beta set bits == 3: 100.0000%
Bitstring file written to outputs/n2_equilibrium_bitstrings_unmasked.txt (61 unique bitstrings)
JSON summary written to outputs/n2_equilibrium_bitstrings_unmasked.json

=== Masked vs unmasked sampling diversity ===
                          

...

  Got 72 unique bitstrings from 1000000 shots in 0.32s.
Total shots: 1000000
Unique bitstrings: 72
Top 10 bitstrings:
  000111000111  count=980009  prob=0.980009
  010101010101  count=4600  prob=0.004600
  001011001011  count=4318  prob=0.004318
  001011000111  count=3037  prob=0.003037
  000111001011  count=2963  prob=0.002963
  000111110100  count=454  prob=0.000454
  110100000111  count=436  prob=0.000436
  000111001101  count=342  prob=0.000342
  001101000111  count=334  prob=0.000334
  010011000111  count=258  prob=0.000258
Fraction of shots on the single most frequent bitstring: 0.980009
Particle-number check (fraction of shots, by weight):
  total set bits == 6: 100.0000%
  alpha set bits == 3: 100.0000%
  beta set bits == 3: 100.0000%
Bitstring file written to outputs/n2_stretched_bitstrings.txt (72 unique bitstrings)
JSON summary written to outputs/n2_stretched_bitstrings.json
--- Sampling UNMASKED circuit ---
Loading circuit from outputs/n2_stretched_lucj_unmasked.qpy ...
  q

  Got 93 unique bitstrings from 1000000 shots in 0.38s.
Total shots: 1000000
Unique bitstrings: 93
Top 10 bitstrings:
  000111000111  count=796389  prob=0.796389
  010101010101  count=52793  prob=0.052793
  001011001011  count=52424  prob=0.052424
  001011010101  count=14487  prob=0.014487
  010101001011  count=14186  prob=0.014186
  011001011001  count=8095  prob=0.008095
  001101010011  count=6761  prob=0.006761
  010011001101  count=6749  prob=0.006749
  100110100110  count=5595  prob=0.005595
  001011100110  count=3925  prob=0.003925
Fraction of shots on the single most frequent bitstring: 0.796389
Particle-number check (fraction of shots, by weight):
  total set bits == 6: 100.0000%
  alpha set bits == 3: 100.0000%
  beta set bits == 3: 100.0000%
Bitstring file written to outputs/n2_stretched_bitstrings_unmasked.txt (93 unique bitstrings)
JSON summary written to outputs/n2_stretched_bitstrings_unmasked.json

=== Masked vs unmasked sampling diversity ===
                           

Transpilation path: preset pass manager with ffsim.qiskit.PRE_INIT pre-init stage.
Post-transpilation: depth=367, two-qubit gates=1176
Masked ansatz variational energy: -108.3329115631
Building UNMASKED LUCJ operator (interaction_pairs=None; comparison only) ...
  Actual n_reps chosen by full-rank factorisation: 18
Building Qiskit circuit (HF prep + UCJ + measure_all) ...
  Circuit built: 12 qubits, depth 3
Pre-transpilation:  depth=3, two-qubit gates=0
Transpilation path: preset pass manager with ffsim.qiskit.PRE_INIT pre-init stage.
Post-transpilation: depth=511, two-qubit gates=1700
Unmasked ansatz variational energy: -108.4459133125

=== Masked vs unmasked LUCJ ansatz comparison ===
                                        masked          unmasked
n_reps                                      18                18
circuit depth                              367               511
two-qubit gate count                      1176              1700
variational energy             -108.33291156

  Got 197 unique bitstrings from 1000000 shots in 0.35s.
Total shots: 1000000
Unique bitstrings: 197
Top 10 bitstrings:
  000111000111  count=874596  prob=0.874596
  001101001101  count=16795  prob=0.016795
  010011010011  count=15766  prob=0.015766
  000111110010  count=11951  prob=0.011951
  110010000111  count=11860  prob=0.011860
  000111010011  count=11623  prob=0.011623
  010011000111  count=11476  prob=0.011476
  010110000111  count=3305  prob=0.003305
  001101000111  count=3270  prob=0.003270
  000111001101  count=3191  prob=0.003191
Fraction of shots on the single most frequent bitstring: 0.874596
Particle-number check (fraction of shots, by weight):
  total set bits == 6: 100.0000%
  alpha set bits == 3: 100.0000%
  beta set bits == 3: 100.0000%
Bitstring file written to outputs/n2_very_stretched_bitstrings.txt (197 unique bitstrings)
JSON summary written to outputs/n2_very_stretched_bitstrings.json
--- Sampling UNMASKED circuit ---
Loading circuit from outputs/n2_very_stretc

  Got 100 unique bitstrings from 1000000 shots in 0.59s.
Total shots: 1000000
Unique bitstrings: 100
Top 10 bitstrings:
  000111000111  count=328835  prob=0.328835
  010011010011  count=81282  prob=0.081282
  011001011001  count=80944  prob=0.080944
  001101001101  count=78949  prob=0.078949
  100110100110  count=61534  prob=0.061534
  010011001101  count=31152  prob=0.031152
  001101010011  count=30769  prob=0.030769
  010011100110  count=17404  prob=0.017404
  100110010011  count=17246  prob=0.017246
  001101100110  count=17025  prob=0.017025
Fraction of shots on the single most frequent bitstring: 0.328835
Particle-number check (fraction of shots, by weight):
  total set bits == 6: 100.0000%
  alpha set bits == 3: 100.0000%
  beta set bits == 3: 100.0000%
Bitstring file written to outputs/n2_very_stretched_bitstrings_unmasked.txt (100 unique bitstrings)
JSON summary written to outputs/n2_very_stretched_bitstrings_unmasked.json

=== Masked vs unmasked sampling diversity ===
         

In [8]:
print(f"{'preset':20s}{'HF':>13s}{'CCSD':>13s}{'CASCI':>13s}{'exact top wt':>13s}")
for name, r in preset_results.items():
    s1 = r["stage1"]
    m = s1.metadata
    exact_top = s1.exact_wavefunction.weight_on_top_determinant if s1.exact_wavefunction else float("nan")
    print(f"{name:20s}{m['hf_energy']:>13.6f}{m['ccsd_energy']:>13.6f}{m['casci_reference_energy']:>13.6f}{exact_top:>13.4f}")

print()
print(f"{'preset':20s}{'masked E err(mHa)':>19s}{'unmasked E err(mHa)':>21s}")
for name, r in preset_results.items():
    s1 = r["stage1"]
    ccsd_e = s1.metadata["ccsd_energy"]
    masked_err = 1000 * (s1.masked_stats.variational_energy - ccsd_e)
    unmasked_err = 1000 * (s1.unmasked_stats.variational_energy - ccsd_e) if s1.unmasked_stats else float("nan")
    print(f"{name:20s}{masked_err:>19.4f}{unmasked_err:>21.4f}")

print()
print(f"{'preset':20s}{'masked uniq':>13s}{'unmasked uniq':>15s}{'masked top1':>13s}{'unmasked top1':>15s}")
for name, r in preset_results.items():
    s2 = r["stage2"]
    masked = s2["masked"]
    unmasked = s2["unmasked"]
    u_uniq = unmasked["unique_bitstrings"] if unmasked else float("nan")
    u_top1 = unmasked["top1_fraction"] if unmasked else float("nan")
    print(f"{name:20s}{masked['unique_bitstrings']:>13d}{u_uniq:>15}{masked['top1_fraction']:>13.4f}{u_top1:>15.4f}")

print()
print(f"{'preset':20s}{'n_adets':>10s}{'n_bdets':>10s}{'product dim':>13s}{'% of full CI':>14s}")
for name, r in preset_results.items():
    diag = r["stage2b"]["variants"][r["stage2b"]["default_transform"]]["diagnostics"]
    print(
        f"{name:20s}{diag['n_alpha_determinants']:>10d}{diag['n_beta_determinants']:>10d}"
        f"{diag['product_space_dimension']:>13d}{diag['product_space_pct_of_full_ci']:>14.4f}"
    )

preset                         HF         CCSD        CASCI exact top wt
n2_equilibrium        -108.867746  -108.945911  -108.946716       0.9393
n2_stretched          -108.587323  -108.801058  -108.811675       0.7705
n2_very_stretched     -108.309601  -108.764548  -108.726795       0.3455

preset                masked E err(mHa)  unmasked E err(mHa)
n2_equilibrium                  46.4318               0.4930
n2_stretched                   150.1446              11.0087
n2_very_stretched              431.6361             318.6344

preset                masked uniq  unmasked uniq  masked top1  unmasked top1
n2_equilibrium                 48             61       0.9941         0.9408
n2_stretched                   72             93       0.9800         0.7964
n2_very_stretched             197            100       0.8746         0.3288

preset                 n_adets   n_bdets  product dim  % of full CI
n2_equilibrium              15        16          240       60.0000
n2_stretched     

## Outputs and next steps

Per run (primary demo uses `n2_cas66*`; the H2 calibration uses
`h2_sto3g*`; the cross-preset section uses `<preset_name>*`), `outputs/`
contains:

- `*.fcidump` - the active-space Hamiltonian.
- `*_lucj.qpy` / `.json` - the **masked** Stage 1 circuit + metadata.
- `*_lucj_unmasked.qpy` / `.json` - the unmasked comparison circuit + metadata (comparison-only).
- `*_bitstrings.txt` / `.json` (+ `*_bitstrings_unmasked.*`) - Stage 2's own
  bitstring + counts artefact, for this notebook's analysis only. **Not**
  what sbd reads.
- `*_adets.txt` / `*_bdets.txt` (+ `*_sbd_export.json`) - **Stage 2B's sbd
  export**: one determinant per line, no header, no counts, sorted ascending
  (HF first). This is what `--adetfile`/`--bdetfile` actually take.
- `*_hf_only_adets.txt` / `*_hf_only_bdets.txt` - the single-determinant
  calibration file (see the H2 section above).

**sbd (Stage 3, external, not implemented here) takes exactly two/three
files: the `.fcidump`, the `*_adets.txt`, and (unless relying on sbd's
beta=alpha default) the `*_bdets.txt`.** Everything else - the Stage 2
bitstrings file, unmasked-ansatz artefacts, and JSON summaries - is
diagnostic output for this notebook, not sbd input.

**Bit ordering for the sbd export is CONFIRMED** (`CONFIG["sbd_bit_transform"]
= "split"`): verified against the sbd repository's own `AlphaDets.txt`
example file - see `stage2b.py`'s module docstring. The alternative
transforms remain available as a fallback only. The Stage 2 bitstring file's
own `CONFIG["bit_order_mode"]` is a separate, still-unconfirmed setting for
that other artefact.

Sampling-quality metrics (Gini, compactness), orbital-ordering permutations,
optimisation loops, and the diagonalisation itself (sbd does that) remain
out of scope for this notebook.